In [ ]:
import os
import sys
import pickle
import numpy as np
from al_automated.config import ActiveLearningConfig
from al_automated.ground_truth_builder import generate_synthetic_truth
from al_automated.loop_controller import ActiveLearningLoop

In [ ]:
# User configures settings

# Create a custom dosage array (e.g., 0.2 to 4.0 with steps of 0.4)
custom_dosages = list(np.round(np.arange(0.2, 4.2, 0.4), 2))

config = ActiveLearningConfig(
    max_cycles=8,            
    ensemble_size=100,        
    budget_circuits=2,       
    budget_dosages=2,
    dist_type='lognormal',
    dosages=custom_dosages
    # Exponential cooling prevents bouncing
)

In [ ]:
# System loads GCAD inputs
sys.path.append(os.path.abspath("GCAD"))

with open("selected_M_circuits.pkl", "rb") as f:
    my_circuits = pickle.load(f)

if isinstance(my_circuits, list):
    # If the list contains tuples (e.g., from a Pareto front output), extract just the Topo object
    if len(my_circuits) > 0 and isinstance(my_circuits[0], tuple):
        my_circuits = {f"Circuit_{i+1}": circ[0] for i, circ in enumerate(my_circuits)}
    else:
        # If it's just a raw list of Topo objects
        my_circuits = {f"Circuit_{i+1}": circ for i, circ in enumerate(my_circuits)}

print(f"Loaded {len(my_circuits)} circuits: {list(my_circuits.keys())}")

# System creates the "Lab Reality" 
# Make changes in al_automated.ground_truth_builder.generate_synthetic_truth()
my_hidden_truth = generate_synthetic_truth(config.nominal_parts_path)

# Execute the Engine
engine = ActiveLearningLoop(my_circuits, my_hidden_truth, config)
calibrated_parts, run_history = engine.run()